# ARIMA & SARIMA 

### MIT License
### loreroma265-max
A time series analysis of US beer, wine and liquor store retail sales 
(NAICS 4453) using ARIMA and SARIMA models.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('data/MRTSSM4453USN.csv', header=0)
df.head()

In [ ]:
df.columns = ['Month', 'Sales']
df['Month'] = pd.to_datetime(df['Month'])
df.set_index('Month', inplace=True)
df.head()

In [ ]:
df.describe()

In [ ]:
df.plot(figsize=(12, 6), title='US Beer, Wine & Liquor Store Sales')
plt.savefig('outputs/01_raw_series.png', dpi=150)
plt.show()

## Adfuller Test

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adfuller_test(sales):
    result = adfuller(sales)
    labels = ['ADF Test Statistic', 'p-value', '# Lags Used', '# Observations']
    for value, label in zip(result, labels):
        print(label + ' : ' + str(value))
    if result[1] <= 0.05:
        print("Strong evidence against null hypothesis \u2014 data is STATIONARY")
    else:
        print("Weak evidence against null hypothesis \u2014 data is NON-STATIONARY")

In [ ]:
adfuller_test(df['Sales'])

## Non-Stationary

The test found that the series is non-stationary meaning we reject the null hypothesis. Therefore, in the following cells, we compute the difference to get a stationary series.

In [ ]:
df['Seasonal First Difference'] = df['Sales'] - df['Sales'].shift(12)
df['Seasonal First Difference'].plot(figsize=(12, 6))
plt.savefig('outputs/02_seasonal_diff.png', dpi=150)
plt.show()

In [ ]:
adfuller_test(df['Seasonal First Difference'].dropna())

In [ ]:
import statsmodels.api as sm

fig = plt.figure(figsize=(12, 8))
ax1 = fig.add_subplot(211)
fig = sm.graphics.tsa.plot_acf(
    df['Seasonal First Difference'].iloc[13:], lags=40, ax=ax1)
ax2 = fig.add_subplot(212)
fig = sm.graphics.tsa.plot_pacf(
    df['Seasonal First Difference'].iloc[13:], lags=40, ax=ax2)
plt.savefig('outputs/03_acf_pacf.png', dpi=150)
plt.show()

## SARIMA

We now run SARIMA with order (1,1,1) and seasonal order (1,1,1,12)

In [ ]:
model = sm.tsa.statespace.SARIMAX(
    df['Sales'], order=(1,1,1), seasonal_order=(1,1,1,12))
results = model.fit()
print(results.summary())

In [ ]:
df['forecast'] = results.predict(start=390, end=410, dynamic=True)
df[['Sales', 'forecast']].plot(figsize=(12, 8))
plt.savefig('outputs/04_forecast_vs_actual.png', dpi=150)
plt.show()

In [ ]:
from pandas.tseries.offsets import DateOffset

future_dates = [df.index[-1] + DateOffset(months=x) for x in range(0, 24)]
future_df = pd.DataFrame(index=future_dates[1:], columns=df.columns)
future_df = pd.concat([df, future_df])

future_df['forecast'] = results.predict(start=411, end=433, dynamic=True)
future_df[['Sales', 'forecast']].plot(figsize=(12, 8))
plt.savefig('outputs/05_future_forecast.png', dpi=150)
plt.show()

# Conclusion

The model achieves a MAPE of around 4.4% on a 24-month test period, 
with a December 2026 sales forecast of approximately $7.7 billion.